# Rendering Demo


In [ ]:
from importlib.metadata import version
from pathlib import Path
import random as rng

import polars as pl
from matplotlib.figure import Figure

import polarstory as ps

N = 250
LABEL = ''  # e.g. 'before' or 'after' when versions are identical
OUTPUT_DIR = Path('reports')


## 1. Generate synthetic data

In [ ]:
df = pl.DataFrame({
    'patient_id': range(1, N + 1),
    'age': [rng.randint(18, 90) for _ in range(N)],
    'sex_admin': [rng.choice(['F', 'M']) for _ in range(N)],
    'encounter_type': [
        rng.choices(['Outpatient', 'Inpatient', 'Emergency'], weights=[65, 20, 15])[0]
        for _ in range(N)
    ],
    'length_of_stay': [
        None if rng.random() < 0.1 else round(rng.uniform(0, 12), 1)
        for _ in range(N)
    ],
})

summary_df = (
    df.group_by('encounter_type')
    .agg(
        count=pl.len(),
        mean_age=pl.col('age').mean(),
        mean_stay=pl.col('length_of_stay').mean(),
    )
    .with_columns(proportion=pl.col('count') / pl.col('count').sum())
    .sort('encounter_type')
)

df.head()

## 2. Build the native report
The same components are added on old and new versions.

In [ ]:
report = ps.Report(
    'Synthetic cohort report',
    subtitle=f'{N} synthetic patients',
)

report.add_text(
    'This report uses simulated data, not actual patient records.\n'
    'Plain text test: <strong>not bold</strong> & ordinary text.',
    title='About this report',
)

with report.section('Native tables'):
    report.add_table(
        summary_df,
        title='Encounter characteristics',
        caption='Aggregated counts and means.',
        notes='Mean length of stay excludes null values.',
        id='native-summary',
    )
    report.add_table(
        df.select('age', 'length_of_stay').describe(),
        title='Numeric summary',
    )

with report.section('Charts'):
    figure = Figure(figsize=(7, 3))
    ax = figure.subplots()
    ax.hist(df.get_column('age').to_list(), bins=12, edgecolor='white')
    ax.set(xlabel='Age (years)', ylabel='Patients')
    figure.tight_layout()
    report.add_chart(figure, title='Age distribution', caption='Synthetic cohort.')


## 3. Add HTML Components

In [ ]:
# detect new methods without masking rendering errors
has_gt = callable(getattr(report, 'add_great_table', None))
has_html = callable(getattr(report, 'add_html', None))

if has_gt:
    from great_tables import GT

    gt = (
        GT(summary_df)
        .tab_header(title='Encounter characteristics', subtitle='Synthetic cohort')
        .cols_label(
            encounter_type='Encounter type',
            count='Patients',
            mean_age='Mean age',
            mean_stay='Mean stay (days)',
            proportion='Percent',
        )
        .fmt_number(columns='count', decimals=0)
        .fmt_number(columns=['mean_age', 'mean_stay'], decimals=1)
        .fmt_percent(columns='proportion', decimals=1)
        .opt_row_striping()
    )
    with report.section('Great Tables'):
        report.add_great_table(gt, notes='Same data as the native summary table.')
else:
    print('add_great_table() unavailable; skipping external table.')

if has_html:
    with report.section('Custom HTML'):
        report.add_html(
            '<div style="padding:12px;border-left:4px solid #388e75;'
            'background:#edf7f2;color:#23443a">'
            '<strong>HTML rendering test</strong>'
            '<p>This is formatted HTML, not escaped text.</p>'
            '</div>',
            title='Custom callout',
            id='html-example',
        )
else:
    print('add_html() unavailable; skipping custom HTML.')


## 4. Save the standalone report
Set `LABEL` above to `before` or `after` if comparing two checkouts with the same version and feature availability.

In [ ]:
ps_version = version('polarstory')
label = LABEL or f'gt{int(has_gt)}-html{int(has_html)}'
path = OUTPUT_DIR / f'polarstory-{ps_version}-{label}.html'
path = report.write_html(path).resolve()

print(f'polarstory {ps_version} | polars {pl.__version__}')
print(f'Loaded from: {ps.__file__}')
print(f'Report saved to: {path}')

from IPython.display import FileLink, display

display(FileLink(str(path.relative_to(Path.cwd()))))

## 5. Preview the report in the notebook
This uses polarstory’s notebook-specific rendering, not the standalone HTML CSS. Compare light and dark notebook themes as well as the saved browser output.

In [ ]:
from polarstory.preview import show_report

show_report(report)